# Renal Evidence Studio

Notebook ini mendokumentasikan workflow proyek CKD screening: mengambil dataset, melatih model, membaca metrik, melihat feature importance, dan mencoba prediksi contoh.

Catatan: proyek ini hanya untuk edukasi dan screening berbasis machine learning, bukan diagnosis medis.

## 1. Setup

Jalankan cell ini dari root repository. Jika dependency belum terpasang, jalankan `pip install -r requirements.txt` terlebih dahulu.

In [1]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path.cwd()
ROOT

ModuleNotFoundError: No module named 'pandas'

## 2. Ambil dan Cache Dataset

Dataset utama berasal dari UCI Chronic Kidney Disease dataset id 336. Script fetch memiliki fallback offline agar notebook tetap bisa dijalankan ketika koneksi tidak tersedia.

In [ ]:
from scripts.fetch_data import ensure_dataset, DATA_PATH, METADATA_PATH

dataset_path = ensure_dataset()
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

print(f"Dataset path: {dataset_path}")
print(f"Rows: {metadata['row_count']}")
print(f"Features: {metadata['feature_count']}")
print(metadata['citation'])

## 3. Eksplorasi Singkat Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df['classification'].value_counts(dropna=False)

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename('missing_count')
    .to_frame()
)
missing_summary.head(12)

## 4. Training dan Evaluasi

Training membandingkan Logistic Regression, Decision Tree, Random Forest, dan SVC. Model terbaik dipilih berdasarkan mean cross-validated F1 untuk label `ckd`.

In [ ]:
from src.train import train_and_save

result = train_and_save(random_state=42)
result['selected_model']

In [ ]:
metrics = result['metrics']
leaderboard = pd.DataFrame(
    [
        {
            'model': model_name,
            'mean_f1': values['mean_f1'],
            'std_f1': values['std_f1'],
        }
        for model_name, values in metrics['cv_scores'].items()
    ]
).sort_values('mean_f1', ascending=False)

leaderboard

In [ ]:
summary_metrics = {
    key: metrics[key]
    for key in ['selected_model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'confusion_matrix']
}
summary_metrics

## 5. Feature Importance

Importance di bawah menjelaskan perilaku model pada data evaluasi. Ini bukan klaim kausal medis.

In [ ]:
feature_importance_path = ROOT / 'app' / 'artifacts' / 'feature_importance.json'
feature_importance = json.loads(feature_importance_path.read_text(encoding='utf-8'))

pd.DataFrame(feature_importance['top_features'])

## 6. Contoh Prediksi Lokal

Cell ini memakai service predictor yang sama dengan API FastAPI.

In [ ]:
from app.schemas import SAMPLE_PAYLOAD
from app.services.predictor import Predictor

predictor = Predictor()
prediction = predictor.predict(SAMPLE_PAYLOAD)
prediction

## 7. Menjalankan Web App

Dari terminal root repository:

```powershell
python -m uvicorn app.main:app --reload --port 8000
```

Lalu buka `http://127.0.0.1:8000` di browser.